# ARIA — Autonomous Research & Iteration Agent
## Training Notebook
**Author:** Angel Singh
**Hackathon:** Meta PyTorch OpenEnv Hackathon × Scaler 2026

This notebook trains an LLM agent to autonomously complete enterprise workflows using GRPO via HuggingFace TRL + Unsloth.

## Step 1 — Install Dependencies

In [ ]:
!pip install openenv unsloth trl transformers gradio fastapi uvicorn matplotlib -q
print('✅ All dependencies installed!')

## Step 2 — Clone Repository

In [ ]:
!git clone https://github.com/yourusername/aria-env
%cd aria-env
print('✅ Repository cloned!')

## Step 3 — Verify Environment

In [ ]:
from environment.aria_env import ARIAEnvironment

env = ARIAEnvironment(capped=True, difficulty=1)
obs = env.reset()
print('✅ Environment working!')
print('Step:', obs['step'])
print('Tasks:', obs['tasks_completed'], '/', obs['total_tasks'])
env.render()

## Step 4 — Test Reward Model

In [ ]:
from environment.reward import RewardModel

reward_model = RewardModel(capped=True)
reward = reward_model.compute(
    tasks_completed=5,
    total_tasks=5,
    tool_calls=20,
    min_tool_calls=20,
    adaptation_triggered=True,
    policy_changed=True,
    action_history=[],
)
print('✅ Reward Model working!')
print('Reward:', reward)
print('Breakdown:', reward_model.get_last_reward_breakdown())

## Step 5 — Run Baseline Demo

In [ ]:
!python demo.py

## Step 6 — Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=True,
)
print('✅ Model loaded!')
print('Model:', model.__class__.__name__)

## Step 7 — Define Reward Function for GRPO

In [ ]:
from environment.aria_env import ARIAEnvironment
from training.train import build_prompt, parse_action

def aria_reward_function(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        env = ARIAEnvironment(capped=True, difficulty=1)
        obs = env.reset()
        total_reward = 0.0
        for _ in range(20):
            action = parse_action(completion)
            obs, reward, done, info = env.step(action)
            total_reward += reward
            if done:
                break
        rewards.append(total_reward)
    return rewards

print('✅ Reward function defined!')

## Step 8 — Train with GRPO

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from environment.aria_env import ARIAEnvironment
from training.train import build_prompt

# Build training prompts
env = ARIAEnvironment(capped=True, difficulty=1)
obs = env.reset()
prompt = build_prompt(obs)

train_dataset = [{"prompt": prompt} for _ in range(50)]

# GRPO Config
grpo_config = GRPOConfig(
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=256,
    num_train_epochs=1,
    logging_steps=5,
    output_dir='./results',
    report_to='none',
)

# Trainer
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=aria_reward_function,
    args=grpo_config,
    train_dataset=train_dataset,
)

print('✅ Trainer ready! Starting training...')
trainer.train()
print('✅ Training complete!')

## Step 9 — Generate Reward Curves

In [ ]:
!python demo.py
print('✅ Graphs saved to results/')

## Step 10 — Save Model to HuggingFace

In [ ]:
from huggingface_hub import login
login(token='')

model.save_pretrained('./aria-trained-model')
tokenizer.save_pretrained('./aria-trained-model')

model.push_to_hub('angel-singh/aria-trained-model')
tokenizer.push_to_hub('angel-singh/aria-trained-model')
print('✅ Model pushed to HuggingFace!')